<a href="https://colab.research.google.com/github/tuankhoin/CO3133-Deep-Learning/blob/main/Week_5_Transformers_%26_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ho Chi Minh University of Technology (HCMUT)

CO3133 - Deep Learning and Its Applications

# Week 5 - Attention and Transformers

### From Tokens and Attention to Modern Transformer Families

This lecture combines four themes:

1. positional information, masking, and training basics;
2. attention and the Transformer core;
3. model families: BERT, GPT, T5, and ViT;
4. practical fine-tuning, efficiency, debugging, and deployment.

## Learning Outcomes

By the end of this lecture, you should be able to:

- Explain how raw data becomes a sequence of Transformer tokens.
- Explain why Transformers need positional information.
- Distinguish absolute, relative, and rotary positional schemes.
- Derive scaled dot-product attention and track tensor shapes.
- Explain multi-head self-attention and a complete Transformer block.
- Compare encoder-only, decoder-only, encoder-decoder, and Vision Transformer families.
- Match masks and training objectives to BERT-, GPT-, and T5-style models.
- Reason about parameter count, compute, and memory cost.
- Design a practical fine-tuning workflow using pretrained Transformers.
- Explain PEFT / LoRA, quantization, KV cache, and common efficiency techniques.

## Lecture Contents

### Part I - Input and Sequence Contract
1. Tokens, tokenization, and embeddings
2. Positional information
3. Masks
4. Training objectives and optimization basics

### Part II - Transformer Core
5. Why attention?
6. Scaled dot-product attention
7. Multi-head attention
8. Transformer block
9. Parameters and complexity

### Part III - Model Families
10. BERT
11. GPT
12. T5
13. Vision Transformer
14. Choosing a family

### Part IV - Practical Transformers
15. Fine-tuning workflow
16. Data and task heads
17. PEFT / LoRA
18. Efficiency and quantization
19. Debugging and deployment

---
# Part I - From a Sequence to Transformer Input
---

## Link to Week 4: RNN $\rightarrow$ Transformer

Last week, an RNN received a sequence of vectors:

$$
\mathbf{x}_1,\mathbf{x}_2,\ldots,\mathbf{x}_T
$$

and processed them **one after another**:

$$
\mathbf{h}_t=f(\mathbf{x}_t,\mathbf{h}_{t-1}).
$$

The recurrence itself tells the RNN that $\mathbf{x}_1$ comes before $\mathbf{x}_2$.

A Transformer removes this recurrence. It wants to process many sequence positions **together** using attention.

So before attention, we must answer two questions:

1. **What vector represents each item?**  
   $\rightarrow$ tokenization + **token embedding**

2. **How does the model know where each item occurs?**  
   $\rightarrow$ **positional information**

This gives the Transformer input:

$$
\boxed{
\text{raw input}
\rightarrow
\text{tokens}
\rightarrow
\text{token IDs}
\rightarrow
\text{token vectors}
+
\text{position}
\rightarrow
X_0
}
$$

## 1. Tokens

A Transformer does not directly understand a sentence. We first split the raw input into **tokens**.

A **token** is one unit occupying one position in the Transformer sequence.

> For text, a simple example could be:
$
[\text{loe},\ \text{hoe},\ \text{chích},\ \text{chòe}]
$
, so the sequence length is:
$
n=4
$

Tokens do **not** have to be whole words. Modern text models often use **subwords**:

$$
learning ⟶ [\text{learn},\ \text{ing}]
$$

<img src="https://www.researchgate.net/publication/370310285/figure/fig1/AS:11431281153923880@1682600732649/Patch-splitting-mechanisms-in-vision-Transformers-a-The-original-image-b-The.png" height=200/>

Other modalities use the same idea:

| Input | Possible token |
|---|---|
| Text | word / subword / character |
| Image | image patch |
| Audio | frame / chunk |
| Video | frame patch / tubelet |

> **Token = one item that attention can interact with.**

## 2. Tokenizer

A **tokenizer** performs the conversion from raw input into the discrete symbols expected by the model.

For text:
$\boxed{
\text{sentence}
\rightarrow
\text{tokens}
\rightarrow
\text{token IDs}
}$

Example:

$$
[\text{deep},\text{learning},\text{is},\text{useful}]
⟶
[6767,\ 6969,\ 2267,\ 3618]
$$

The integers are **IDs**, not meaningful numerical values (example, ID `6767` is not "larger" or "better" than ID `3618`).

The tokenizer uses a **vocabulary**:

$$
\text{token}
\leftrightarrow
\text{integer ID}.
$$

Special tokens may also be added, such as:

- `[PAD]` — padding
- `[CLS]` — sequence/classification token in some models
- `[SEP]` — separator
- beginning/end-of-sequence tokens

> The tokenizer must match the pretrained model, because the model learned meanings for that tokenizer's IDs.

## Tokenization Choices

Different tokenization strategies trade sequence length against vocabulary size.

| Method | Example | Main trade-off |
|---|---|---|
| Word-level | `['Bản', 'lĩnh', 'thì', 'đến', '66B', 'Nguyễn', 'Sỹ', 'Sách']` | intuitive, but huge vocabulary / unknown words |
| Character-level | `['B', 'ả', 'n', ' ', 'l', 'ĩ', 'n', 'h',..., '6', '6', 'B',...]` | tiny vocabulary, but very long sequences |
| Subword | `['B', 'ản', 'l', 'ĩnh',..., '66', 'B',...]` | good compromise; common in modern Transformers |

Common subword systems include:

- BPE
- WordPiece
- SentencePiece

### Why sequence length matters

If tokenization creates more tokens, $n$ becomes larger ⟶ Self-attention later constructs an $n\times n$ relation matrix
> longer sequences == more memory and computation.

## 3. Token IDs Are Not Yet Useful Neural-Network Inputs

IDs only tell us **which vocabulary entries** appeared.
> We should not feed them directly as numerical magnitudes:
$
99 \not> 89
$


Instead, each ID is used to **look up a learnable vector**.

That vector is called a **token embedding**.


$$
\boxed{
\text{token ID}
\rightarrow
\text{embedding lookup}
\rightarrow
\text{dense vector}
}
$$

## 4. Token Embedding

<img src="https://miro.medium.com/v2/resize:fit:1400/1*-LJr59LJCTLL4YJodiJsVA.png" height=200/>

An **embedding** is a dense vector representation learned for a discrete item.

Suppose the vocabulary contains $V$ possible tokens and each token is represented using $d$ numbers.

The embedding table is:

$$
E
\in
\mathbb{R}^{V\times d}.
$$

For token ID $i$, we select row $i$:

$$
\mathbf{e}_i=E[i]
\in
\mathbb{R}^{d}.
$$

Example:

$$
\text{'Trường Con'}
\rightarrow
226789
\rightarrow
\begin{bmatrix}
0.18 & -0.36 & 0.35 & \cdots
\end{bmatrix}.
$$

During training, the model adjusts these numbers so useful tokens receive useful representations.

### Intuition

Token ID = **name / index**  
Embedding = **what the neural network knows about that token as a vector**

For a batch $T$:

$$\boxed{
T\in\mathbb{N}^{B\times n}
\quad\rightarrow\quad
E_{\text{token}}(T)
\in
\mathbb{R}^{B\times n\times d_{\text{model}}}}
$$

## Embeddings Connect Back to Earlier Neural Networks

In previous weeks, neural networks expected numerical feature vectors such as:

$$
\mathbf{x}\in\mathbb{R}^{D}.
$$

Token embedding simply gives us such a vector for every discrete token:

$$
\text{token}_1
\rightarrow
\mathbf{x}_1,
\qquad
\text{token}_2
\rightarrow
\mathbf{x}_2,
\qquad \ldots
$$

Therefore, after embedding, our sentence becomes a matrix:

$$\boxed{
X_{\text{token}}
=
\begin{bmatrix}
- & \mathbf{x}_1 & -\\
- & \mathbf{x}_2 & -\\
& \vdots &\\
- & \mathbf{x}_n & -
\end{bmatrix}
\in
\mathbb{R}^{n\times d_{\text{model}}}}
$$

This is now suitable for neural-network operations.

But one problem remains:

> **Where is each token located in the sequence?**

## 5. Why Is Position a Separate Problem?

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/ordering.jpg" height=300/>

Consider: `dog bites man` and `man bites dog`

- Same tokens, but order changes the meaning.

An RNN naturally sees order because it processes in sequence:

$$
\mathbf{x}_1
\rightarrow
\mathbf{x}_2
\rightarrow
\mathbf{x}_3
$$

Self-attention, however, can process all token vectors together.
- Without **extra position information**, attention mainly sees token content and does not inherently know whether a vector came from position 1, 2, or 3.

So we add a representation of **where the token is**. This is called **positional encoding / positional embedding**.

## 6. Token Information + Position Information

<img src="https://substackcdn.com/image/fetch/$s_!TvNT!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F7b69f56c-99ee-4132-83c0-ea67fa4c1c58_1925x2077.png" height=300/>

For each sequence position $i$, we have:

- token embedding $\mathbf{e}_i$ — **what is here?**
- position vector $\mathbf{p}_i$ — **where is it?**

A simple Transformer input is:

$$\boxed{
\mathbf{x}_i
=
\mathbf{e}_i+\mathbf{p}_i}
$$

For the whole batch:

$$\boxed{
X_0
=
E_{\text{token}}
+
E_{\text{pos}}
\in
\mathbb{R}^{B\times n\times d_{\text{model}}}}
$$

The dimensions must match because the two vectors are added.

### Mental model

$$
\boxed{
\text{Transformer input}
=
\text{content}
+
\text{position}
}
$$

Now each sequence position contains both **what** the token is and **where** it occurs.

## 7. Positional Encoding

**Positional encoding** means injecting sequence-position information into the model.

There are several ways to do this.

## 7.1 Absolute position encoding

Represent: token 0, token 1, token 2, ... directly.

Two common approaches:

1. **Learned positional embedding** — learn one vector for each position.
2. **Sinusoidal encoding** — calculate a fixed vector from sine/cosine functions.

Later approaches such as **relative position bias** and **RoPE** focus more directly on relationships such as:

> "token $j$ is 3 positions away from token $i$."

The goal is the same:

> **make order visible to attention.**


### 7.1.1 Learned absolute positions

Store a trainable position table:

$$\boxed{
P
\in
\mathbb{R}^{n_{\max}\times d}}
$$

For a sequence of length $n$:

$$\boxed{
X_0
=
E_{\text{token}}
+
P[0:n]}
$$

Additional parameters:

$$
n_{\max}d
$$


## 7.1.2 Sinusoidal Positional Encoding

Instead of learning the table, calculate each position vector using fixed sine and cosine functions.

So both approaches provide a vector $\mathbf{p}_i$ for position $i$; they differ in **where that vector comes from**.

For position $\text{po}$ and dimension pair $2i,2i+1$:

$$\boxed{
PE_{(\text{po},2i)}
=
\sin
\left(
\frac{\text{po}}
{10000^{2i/d}}
\right)\\
PE_{(\text{po},2i+1)}
=
\cos
\left(
\frac{\text{po}}
{10000^{2i/d}}
\right)}
$$

You do **not** need to think of this as "the model understands sine waves."

Instead:

- every position receives a different vector;
- different dimensions change at different rates;
- the pattern gives the model information from which relative/absolute locations can be inferred;
- no position parameters need to be learned.

## 7.2 Relative Position Bias and RoPE

<img src="https://miro.medium.com/1*ou3SzkHS2uZCR8ewwI75QA.png" height=300/>

Absolute encoding says roughly:

> "This token is at position 17."

Modern designs often care about relative relationships:

> "This token is 2 positions after that token."

### 7.2.1 Relative position bias

<img src="https://i.pinimg.com/736x/07/c6/dc/07c6dc9f7d0439ae7c2e2866cc975af4.jpg" height=200/>

Modify the attention score to add in $B_{rel}$:

$$\boxed{
S
=
\frac{QK^\top}{\sqrt{d_k}}
+
B_{\text{rel}}}
$$

So distance directly influences which tokens attend to each other.

### 7.2.2 RoPE (rotary positional embeddings)


<img src="https://assets.insightmediagroup.io/media/wp-content/uploads/2024/07/1EWz8ImltNHpDjMB8bOq_tQ.png" height=300/>

RoPE rotates $Q$ and $K$ according to token positions before computing attention:

$$\boxed{
Q'=R_{\text{pos}}Q\\
K'=R_{\text{pos}}K}
$$

The resulting dot products encode relative positional relationships.

For now, remember the distinction:

| Method | Main intuition |
|---|---|
| Absolute | give each position its own position information |
| Relative / RoPE | make pairwise token relationships position-aware |

## Putting Together

$$
\boxed{
\begin{aligned}
\text{Raw text}
&\rightarrow \text{Tokens}\\
&\rightarrow \text{Token IDs}\\
&\rightarrow \text{Token embeddings}\\
&\rightarrow +\text{Position information}\\
&\rightarrow X_0\in\mathbb{R}^{B\times n\times d_{\text{model}}}
\end{aligned}
}
$$

At this point:

- each row/token vector says **what is present**;
- position information says **where it is**;
- all positions have a common vector size $d_{\text{model}}$.

Now the Transformer can ask the next question:

> **Which other tokens should each token pay attention to?**

That leads to **attention masks** and then the attention mechanism itself.

## 5. Masks

<img src="https://miro.medium.com/0*AZ9q71zKfSVgw0Qe.png" height=200/>

Masks define **which attention connections are legal**.

Two important types:

- Padding mask: Hide PAD tokens introduced when batching variable-length sequences.
- Causal mask: Prevent token $t$ from attending to future positions (j>t). Causal masking is required for autoregressive next-token prediction.

## Masking in Attention Scores

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/masked.jpg" height=300/>

Attention with a mask:

$$\boxed{
A
=
\operatorname{softmax}
\left(
\frac{QK^\top}{\sqrt{d_k}}
+
M
\right)}
$$

Typical mask entries:

$$
M_{ij}=0
\quad
\text{if attention is allowed},
$$

$$
M_{ij}=-\infty
\quad
\text{if attention is blocked}.
$$

Important:

> Apply the mask **before softmax**.

After softmax, blocked positions receive attention weight approximately zero.

## Mask Shapes

Depending on implementation, a mask may use shapes such as `(B,1,n,n)` to broadcast across attention heads, or `(B,n,n)`

For padding-only key masking, a compact form may be broadcast from: `(B,1,1,n)`

Shape discipline is essential: many Transformer bugs are simply incorrect broadcasting or mask direction.

## 6. Common Transformer Training Objectives

| Objective | Prediction | Typical masking |
|---|---|---|
| Autoregressive LM | next token $x_t$ from $x_{<t}$ | causal + padding |
| Masked LM | masked tokens from visible context | padding, no causal mask |
| Encoder-decoder / seq2seq | target token $y_t$ from source + $y_{<t}$ | encoder padding + decoder causal + cross-attention mask |

The training objective and attention pattern must agree.

## Autoregressive Language Modeling

Predict each token from previous tokens:

$$
p(x_t\mid x_{<t}).
$$

Loss:

$$
\mathcal{L}
=
-
\sum_{t=1}^{n}
\log
p(x_t\mid x_{<t}).
$$

Training requires:

- causal masking;
- correct input/label shifting;
- ignoring padding positions in the loss.

## Masked Language Modeling
<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/masked.gif" height=300/>

Randomly hide selected input tokens and predict them from surrounding context:

$$
p(x_{\text{masked}}
\mid
x_{\text{visible}}).
$$

The model can use context from both sides of a masked position.

This is naturally suited to **bidirectional encoder models** such as BERT.

## Training Basics

A common Transformer optimization recipe includes:

- AdamW;
- learning-rate warmup;
- learning-rate decay;
- dropout and weight decay;
- gradient clipping;
- mixed precision;
- validation-based checkpoint selection.

Warmup gradually increases the learning rate:

$$\boxed{
\eta_t
=
\eta_{\max}
\frac{t}{T_{\text{warmup}}}}
$$

Then the learning rate is decayed.

## Complexity of the Sequence Contract

For sequence length $n$ and hidden dimension $d$:

- projections: approximately $O(nd^2)$;
- attention score/value operations: approximately $O(n^2d)$;
- FFN: approximately $O(ndd_{\text{ff}})$.

For multi-head attention:

$$
A
\in
\mathbb{R}^{B\times h\times n\times n}.
$$

Thus attention memory scales approximately as

$$
O(Bhn^2).
$$

> Long sequence length is often the dominant memory problem.

---
# Part II - Attention and the Transformer Core
---

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/triggered.jpg" height=500/>

## 7. Why Attention?

### RNN limitation

Information from far-away positions must travel through many recurrent steps, and token-by-token computation limits parallelism.

### CNN limitation

Convolutions are parallel, but local receptive fields require deeper stacks to connect distant positions.

### Attention idea

At one layer, each token can directly read information from other tokens in the sequence.

## Shape Discipline

Core Transformer representation:

$$
X
\in
\mathbb{R}^{B\times n\times d_{\text{model}}}.
$$

Notation:

- $B$: batch size
- $n$: token count
- $d_{\text{model}}$: hidden dimension
- $d_k,d_v$: per-head key/value dimensions
- $d_{\text{ff}}$: FFN hidden dimension

Across a Transformer stack, block input and output usually remain:

$$
(B,n,d_{\text{model}}).
$$

This enables residual addition.

## 8. Query, Key, Value

<img src="https://kodekloud.com/kk-media/image/upload/v1752879144/notes-assets/images/Introduction-to-OpenAI-Understanding-Attention-Mechanisms-in-Transformers/attention-mechanism-transformers-diagram.jpg" height=300/>

From token representations $X$, learn three projections:

$$
Q=XW_Q,
\qquad
K=XW_K,
\qquad
V=XW_V.
$$

Human intuition:

- **Query:** what information am I looking for?
- **Key:** what information do I contain / how should I be matched?
- **Value:** what information should be passed if I am selected?

Self-attention means $Q,K,V$ come from the **same sequence**.

## Scaled Dot-Product Attention

Compute attention scores:

$$\boxed{
S
=
\frac{QK^\top}{\sqrt{d_k}}}
$$

Normalize each row:

$$
A
=
\operatorname{softmax}(S).
$$

Mix the values:

$$
Y
=
AV.
$$

For one head:

$$
Q,K,V
\in
\mathbb{R}^{B\times n\times d_k},
$$

$$
A
\in
\mathbb{R}^{B\times n\times n}.
$$

## Why Divide by $\sqrt{d_k}$?

- As $d_k$ grows, raw dot products can become large in magnitude.
- Large logits make softmax very sharp and can reduce useful gradients.
- The scaling
$
\frac{1}{\sqrt{d_k}}
$
keeps attention-score magnitudes in a more stable range.

## Attention as a Weighted Average

For token $i$:

$$\boxed{
\mathbf{y}_i
=
\sum_{j=1}^{n}
\alpha_{ij}\mathbf{v}_j}
$$

where

$$
\alpha_{ij}\ge0,
\qquad
\sum_j\alpha_{ij}=1.
$$

Interpretation:

- $\alpha_{ij}$ measures how much token $i$ attends to token $j$.
- attention recombines **value vectors**, not raw tokens.

## Shape Flow Through Attention

Typical flow:

$$
X
\rightarrow
Q,K,V
$$

$$
(B,n,d_{\text{model}})
\rightarrow
(B,n,d_k)
$$

$$
QK^\top
\rightarrow
(B,n,n)
$$

$$
AV
\rightarrow
(B,n,d_v)
$$

$$
\text{output projection}
\rightarrow
(B,n,d_{\text{model}}).
$$

Only intermediate tensors change shape.

## 9. Multi-Head Self-Attention

<img src="https://i.redd.it/5oqqn2l5vgfg1.jpeg" height=300/>

Instead of one attention operation, use $h$ heads in parallel.

Typical per-head dimensions:

$$
d_k=d_v=\frac{d_{\text{model}}}{h}.
$$

Each head has its own attention matrix $
A
\in
\mathbb{R}^{B\times h\times n\times n}
$

Human intuition:

> Multiple heads are several simultaneous "ways of looking" at relationships in the same sequence.

## Multi-Head Attention Computation

For each head:

$$\boxed{
\text{head}_i
=
\operatorname{Attention}
(
XW_i^Q,
XW_i^K,
XW_i^V
)}
$$

Concatenate:

$$
H
=
\operatorname{Concat}
(
\text{head}_1,\ldots,\text{head}_h
).
$$

Output projection:

$$
\operatorname{MHA}(X)
=
HW_O.
$$

## MHA Parameter Count

Standard full projection matrices:

$$\boxed{
W_Q,W_K,W_V,W_O
\in
\mathbb{R}^{d_{\text{model}}\times d_{\text{model}}}}
$$

Ignoring biases:

$$\boxed{
\#\text{MHA params}
=
4d_{\text{model}}^2}
$$

With biases:

$$\boxed{
4d_{\text{model}}^2
+
4d_{\text{model}}}
$$

Softmax and masking have no learned parameters.

## 10. Transformer Block

<img src="https://www.researchgate.net/profile/Zhe-Zhou-32/publication/355391815/figure/fig2/AS:1080500143820800@1634622717174/The-standard-transformer-block.png" height=300/>

A Transformer encoder block alternates:

1. **token mixing** using multi-head self-attention;
2. **channel mixing** using an FFN.

It also uses:

- residual / skip connections;
- LayerNorm;
- dropout.

Conceptually:

$$\boxed{
X
\rightarrow
\text{MHA}
\rightarrow
\text{Residual + Norm}
\rightarrow
\text{FFN}
\rightarrow
\text{Residual + Norm}
\rightarrow
Y}
$$

## Feed-Forward Network (FFN)

The FFN operates independently on each token:

$$
\operatorname{FFN}(x)
=
W_2\phi(W_1x+b_1)+b_2.
$$

Typical dimensions:

$$
d_{\text{model}}
\rightarrow
d_{\text{ff}}
\rightarrow
d_{\text{model}}.
$$

Parameter count without biases:

$$\boxed{
2d_{\text{model}}d_{\text{ff}}}
$$

If $
d_{\text{ff}}\approx4d_{\text{model}},
$ then the FFN contributes approximately $
8d_{\text{model}}^2
$ parameters.

## Residual Connections and LayerNorm

- Residual connection helps preserve information and gradient flow.
- LayerNorm stabilizes feature scale within each token representation.

Together they make deep Transformer stacks easier to optimize.

## Pre-Norm vs Post-Norm

### Post-norm

$$
x
\rightarrow
\text{Sublayer}
\rightarrow
+x
\rightarrow
\boxed{\text{LayerNorm}}
$$

### Pre-norm

$$
x
\rightarrow
\boxed{\text{LayerNorm}}
\rightarrow
\text{Sublayer}
\rightarrow
+x
$$

Pre-norm is commonly easier to optimize in deeper Transformer stacks.

## Parameters per Encoder Layer

Ignoring biases and the small LayerNorm parameter count:

$$\boxed{
\#\text{params}
\approx
4d_{\text{model}}^2
+
2d_{\text{model}}d_{\text{ff}}}
$$

If
$
d_{\text{ff}}=4d_{\text{model}}
$
then:

$$\boxed{
\#\text{params}
\approx
12d_{\text{model}}^2}
$$

Most parameters come from linear projections.

## Where Transformer Computation Goes

| Operation | Approximate cost |
|---|---|
| Q/K/V projections | $O(nd^2)$ |
| Attention matrix | $O(n^2d)$ |
| FFN | $O(ndd_{\text{ff}})$ |

Thus:

- large $n$ → attention dominates;
- large $d$ or $d_{\text{ff}}$ → projections / FFN dominate.

## CNN vs Attention

<img src="https://miro.medium.com/1*XxxBgSdr2RbvTAptvH7Agw.png" height=300/>

### CNN

- strong locality bias;
- efficient spatial computation;
- long-range relationships emerge through depth.

### Attention

- weaker built-in locality bias;
- direct long-range token interaction;
- flexible data-dependent mixing.

A useful mental model:

> CNN = stronger inductive bias + efficiency  
> Attention = flexibility + direct global interaction

## PyTorch Transformer APIs

Common modules:

```python
nn.MultiheadAttention
nn.TransformerEncoderLayer
nn.TransformerEncoder
```

Example:

```python
layer = nn.TransformerEncoderLayer(
    d_model=512,
    nhead=8,
    dim_feedforward=2048,
    batch_first=True
)

encoder = nn.TransformerEncoder(
    layer,
    num_layers=6
)
```

Always verify tensor layouts and mask conventions.

---
# Part III - Transformer Model Families
---

## 11. Family Map

| Family | Topology | Attention pattern | Typical use |
|---|---|---|---|
| BERT | Encoder-only | Bidirectional | understanding, classification, extraction |
| GPT | Decoder-only | Causal | generation, instruction following |
| T5 | Encoder-decoder | bidirectional encoder + causal decoder | source-to-target generation |
| ViT | Encoder-only over patches | global / bidirectional | image classification, vision backbone |

> The key difference is **which stack, which mask, and which objective**.

## Encoder-Only vs Decoder-Only vs Encoder-Decoder

### Encoder-only

$$\boxed{
X
\rightarrow
\text{Encoder stack}
\rightarrow
H}
$$

### Decoder-only

$$\boxed{
x_t
\sim
p(x_t\mid x_{<t})
}$$

### Encoder-decoder

$$\boxed{
X_{\text{src}}
\rightarrow
H_{\text{enc}}\\
y_t
\sim
p(y_t\mid y_{<t},H_{\text{enc}})}
$$

The topology determines information flow.

## Masking Defines Family Behavior

| Family | Main mask behavior |
|---|---|
| BERT | valid tokens attend bidirectionally |
| GPT | causal + padding |
| T5 encoder | bidirectional source attention |
| T5 decoder | causal target attention + cross-attention |
| ViT | usually global attention over valid patches |

Same attention equation, different legal connections.

## 12. BERT: Core Idea

<img src="https://ik.imagekit.io/upgrad1/abroad-images/imageCompo/images/Gemini_Generated_Image_h3iov5h3iov5h3iX3X0YV.jpeg" height=300/>

BERT is an **encoder-only** Transformer.

$$\boxed{
X
\in
\mathbb{R}^{B\times n\times d}
\rightarrow
\text{Encoder layers}
\rightarrow
H
\in
\mathbb{R}^{B\times n\times d}}
$$

Attention is bidirectional:

> token $i$ can attend to valid tokens before and after it.

BERT is designed primarily for contextual understanding.

## BERT Pretraining and Downstream Tasks

Masked Language Modeling:
$
p(x_{\text{masked}}
\mid
x_{\text{visible}})
$

The original BERT also used Next Sentence Prediction, while later models may modify or omit it.

Downstream uses include:

- classification;
- named-entity recognition;
- extractive QA;
- retrieval / representation learning.

Sequence-level tasks often use a pooled or `[CLS]` representation.

## 13. GPT: Core Idea

<img src="https://media.geeksforgeeks.org/wp-content/uploads/20260324090208535998/Decoder-only.webp" height=300/>

GPT is a **decoder-only** Transformer.

At position $t$:

$$\boxed{
X_{\le t}
\rightarrow
\text{causal decoder stack}
\rightarrow
p(x_{t+1}\mid x_{\le t})}
$$

Causal attention means token $t$ can only attend to positions
$
\le t
$

This is naturally suited to autoregressive generation.

## GPT Training and Decoding

Training:

$$\boxed{
\mathcal{L}
=
-
\sum_{t=1}^{n}
\log
p(x_t\mid x_{<t})}
$$

Generation proceeds token by token.

Common decoding options:

- greedy;
- beam search;
- top-$k$ sampling;
- top-$p$ / nucleus sampling;
- temperature scaling.

## GPT and KV Cache

Naive generation repeatedly recomputes previous keys and values.

KV cache stores:
$
K_{\le t},
V_{\le t}.
$

At the next step, compute only the new token's projections.

> KV cache is essential for efficient autoregressive inference.

## 14. T5: Encoder-Decoder

<img src="https://substackcdn.com/image/fetch/$s_!aTz3!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F82a9585b-9f04-429c-a59b-8b1afc8e9a0f_1600x1462.png" height=300/>

T5 uses:

$$
X_{\text{source}}
\rightarrow
\text{Encoder}
\rightarrow
H_{\text{enc}}.
$$

Decoder prediction:

$$
p(y_t\mid y_{<t},H_{\text{enc}}).
$$

T5 uses a unified text-to-text formulation:
$
\text{text input}
\rightarrow
\text{text output}.
$

Typical applications include translation, summarization, QA, and structured generation.

## T5 Attention Structure

- Encoder self-attention: Bidirectional over source tokens.
- Decoder self-attention: Causal over generated target tokens.
- Cross-attention: Decoder queries attend to encoder keys and values $
Q_{\text{dec}}K_{\text{enc}}^\top$
  - This combines generated-prefix information with the encoded source.

## 15. Vision Transformer (ViT)

<img src="https://media.geeksforgeeks.org/wp-content/uploads/20250108160032710219/Vision-Transformer-Architecture_.webp" height=300/>

ViT converts an image into a token sequence.

- For $X
\in
\mathbb{R}^{H\times W\times C}$, split into $P\times P$ patches.

Number of patches:

$$\boxed{
N
=
\frac{HW}{P^2}}
$$

Each patch is projected to:
$
\mathbb{R}^{d_{\text{model}}}
$

The patch sequence is processed by a Transformer encoder.

## ViT Patch Embedding

A convolution can perform patch extraction and projection:

```python
patch_embed = nn.Conv2d(
    in_channels=C,
    out_channels=d_model,
    kernel_size=P,
    stride=P
)
```

For a $224\times224$ image with $P=16$:

$$
N
=
\frac{224\cdot224}{16^2}
=
196.
$$

After flattening:

$$
(B,d,14,14)
\rightarrow
(B,196,d).
$$

## Choosing a Transformer Family

Ask:

- Need contextual understanding? → **encoder-only**
- Need open-ended autoregressive generation? → **decoder-only**
- Need explicit source-to-target mapping? → **encoder-decoder**
- Image represented as patches? → **Vision Transformer / vision backbone**

The central question is the required **information flow**.

---
# Part IV - Practical Fine-Tuning and Efficiency
---

## 16. What Is Fine-Tuning?

<img src="https://raw.githubusercontent.com/tuankhoin/CO3057-Computer-Vision/refs/heads/main/assets/w10transfer.jpg" height=300/>

Fine-tuning adapts a pretrained Transformer to a target task:

$$
\text{pretrained model}
+
\text{task data}
+
\text{task head/objective}
\rightarrow
\text{adapted model}.
$$

Fine-tuning is a contract among:

- tokenizer / processor;
- model;
- data;
- task head;
- loss;
- evaluation metric.

## Typical Fine-Tuning Workflow

1. Choose pretrained checkpoint and matching tokenizer / processor.
2. Define task formulation and output format.
3. Build dataset and split.
4. Define padding, truncation, and collate logic.
5. Add or replace the task head.
6. Choose optimization recipe.
7. Train and save best validation checkpoint.
8. Evaluate using task-relevant metrics.
9. Export/deploy with the same preprocessing contract.

## Fine-Tuning Modes

| Mode | What is trained | Main trade-off |
|---|---|---|
| Linear probing | task head only | fast, limited adaptation |
| Full fine-tuning | all parameters | strongest adaptation, expensive |
| Partial unfreezing | upper layers + head | balanced |
| PEFT / LoRA | small added parameters | memory-efficient |

A practical default depends on model size, data size, and hardware.

## 17. Task Formulation Determines the Head

| Task | Output | Typical objective |
|---|---|---|
| Classification | class logits | CrossEntropy |
| Regression | scalar/vector | MSE / MAE |
| Token classification | logits per token | token CE |
| Extractive QA | start/end logits | two output heads |
| Causal LM | vocabulary logits per step | LM CE |
| Seq2seq | target token sequence | decoder LM loss |

Define target shape and loss before choosing model APIs.

## Input-Output Shape Contract

### Text classification

$$
\text{input IDs, attention mask}
\in
\mathbb{N}^{B\times n}
$$

$$
\text{logits}
\in
\mathbb{R}^{B\times C}.
$$

### Token classification

$$
\text{logits}
\in
\mathbb{R}^{B\times n\times C},
$$

$$
\text{labels}
\in
\mathbb{N}^{B\times n}.
$$

Writing tensor shapes before coding prevents many bugs.

## Checkpoint-Tokenizer Contract

Checkpoint and tokenizer must agree on:

- vocabulary;
- special tokens;
- padding side;
- maximum sequence length;
- preprocessing conventions.

A tokenizer from a different checkpoint may technically run while silently damaging performance.

## Dataset and Padding Strategy

| Strategy | Idea | Comment |
|---|---|---|
| Static | pad to global max | simple but wasteful |
| Dynamic | pad to batch max | strong default |
| Bucketing | group similar lengths | improves efficiency |
| Packing | combine short sequences | useful for LM workflows |

Padding strategy directly affects memory and speed.

## Truncation Is a Modeling Decision

If maximum length is too short, important evidence may be removed.

Possible solutions:

- increase max length;
- sliding window;
- chunk + hierarchical pooling;
- use a long-context architecture.

> Truncation can create hidden label noise.

## Fine-Tuning Optimization Checklist

Common choices:

- AdamW;
- warmup + decay;
- dropout;
- weight decay;
- label smoothing where appropriate;
- gradient clipping;
- FP16 / BF16 mixed precision;
- best-checkpoint selection by validation metric.

Large pretrained models usually use smaller learning rates than models trained from scratch.

## Practical Learning-Rate Ranges

| Scenario | Typical starting range |
|---|---|
| Full fine-tuning BERT-like encoder | $10^{-5}$ to $5\times10^{-5}$ |
| Classifier head only | $10^{-4}$ to $10^{-3}$ |
| LoRA / adapter tuning | $10^{-4}$ to $3\times10^{-4}$ |
| Large decoder fine-tuning | conservative; task/hardware dependent |

These are starting points, not universal constants.

## Evaluation Must Match the Task

| Task | Useful metrics |
|---|---|
| Classification | accuracy, macro-F1, weighted-F1 |
| Imbalanced classification | macro-F1, per-class recall |
| NER / token classification | entity-level F1 |
| QA | exact match, F1 |
| Generation | ROUGE, BLEU, BERTScore, human evaluation |
| Retrieval | Recall@$K$, MRR, nDCG |

Loss alone is not enough.

## 18. Parameter-Efficient Fine-Tuning (PEFT)

PEFT changes only a small part of the model.

| Method | Main idea |
|---|---|
| Adapters | insert small bottleneck modules |
| Prefix tuning | learn prefix key/value vectors |
| Prompt tuning | learn soft prompt embeddings |
| LoRA | low-rank updates to linear layers |

PEFT reduces trainable parameter count and optimizer-state memory.

## LoRA: Low-Rank Adaptation

<img src="https://miro.medium.com/1*ys1LNhxDlD7yfFTrcWufGw.png" height=300/>

Original layer:

$$
y=Wx.
$$

LoRA freezes $W_0$ and learns:

$$\boxed{
W'
=
W_0+\Delta W\\
\Delta W
=
BA}
$$

with
$
A
\in
\mathbb{R}^{r\times d_{\text{in}}},
B
\in
\mathbb{R}^{d_{\text{out}}\times r},
$
and
$
r
\ll
d_{\text{in}},d_{\text{out}}.
$

## LoRA Parameter Count

Full layer:
$\boxed{
d_{\text{out}}d_{\text{in}}
}$
parameters.

LoRA:
$\boxed{
r(d_{\text{in}}+d_{\text{out}})
}$
trainable parameters.

- Example:

$$
d_{\text{in}}
=
d_{\text{out}}
=
4096,\\
r=8,
$$

gives
$
65,536
$
LoRA parameters instead of
$
4096^2
\approx
16.8\text{ million}.
$

## 19. Where Fine-Tuning Memory Goes

Training memory includes:

- parameters;
- gradients;
- optimizer states;
- backward activations;
- attention matrices;
- padding waste.

For attention:

$$
A
\in
\mathbb{R}^{B\times h\times n\times n}.
$$

Reducing sequence length can therefore produce very large savings.

## Efficiency Levers

Common techniques:

- dynamic padding and bucketing;
- FP16 / BF16 mixed precision;
- gradient accumulation;
- gradient checkpointing;
- PEFT / LoRA;
- quantization;
- FlashAttention / fused kernels;
- distillation;
- smaller checkpoints.

> Efficiency is a system-level design problem.

## Gradient Accumulation

If a large batch does not fit in GPU memory:

$$
B_{\text{effective}}
=
B_{\text{micro}}
\times
N_{\text{accum}}.
$$

Gradients from several micro-batches are accumulated before the optimizer update.

This raises effective batch size without increasing peak activation memory per micro-batch.

## Gradient Checkpointing

Instead of storing every intermediate activation:

- store selected activations;
- recompute others during backward.

Trade-off:
$
\text{lower memory}
\leftrightarrow
\text{more compute}.
$

Useful for large models and long sequences.

## 20. Quantization

<img src="https://i.redd.it/mj9a20fyvkze1.png" height=300/>

Represent weights and/or activations at lower precision:

$$
\text{FP32}
\rightarrow
\text{FP16/BF16}
\rightarrow
\text{INT8}
\rightarrow
\text{4-bit}.
$$

Benefits:

- less memory;
- potentially faster inference;
- larger models fit on smaller accelerators.

Caution:

> Always evaluate quality after quantization.

## QLoRA-Style Idea

Use:

$$
\text{quantized base model}
+
\text{LoRA adapters}.
$$

The base model remains quantized/frozen while small adapters are trained.

This combines low memory use with low trainable parameter count.

## KV Cache During Generation

For decoder-only inference, cache:
$
K_{\le t},
V_{\le t}.
$

Then only the new token's query/key/value need to be computed.

This substantially improves autoregressive generation efficiency.

## 21. Common Failure Modes

Common failures:

- tokenizer/checkpoint mismatch;
- padding tokens included in attention or loss;
- wrong label shape;
- wrong causal mask direction;
- data leakage;
- excessive truncation;
- too-large model for small data;
- misleading metrics on imbalanced labels.

Many failures are **contract failures**, not architecture failures.

## Overfit-One-Batch Test

Before full training, try to overfit a tiny batch.

### If loss cannot decrease

Suspect:

- data;
- labels;
- head;
- loss;
- mask;
- optimizer.

### If tiny batch overfits but validation fails

Suspect:

- overfitting;
- data quality;
- generalization;
- distribution mismatch.

This is one of the fastest debugging tests.

## 22. Deployment Checklist

Before deployment:

- freeze tokenizer / processor;
- freeze preprocessing;
- export correct model config;
- benchmark latency and throughput on target hardware;
- test edge cases and long inputs;
- monitor distribution shift.

Possible optimizations:

- smaller checkpoint;
- quantization;
- batching;
- KV cache;
- ONNX / TensorRT / optimized serving;
- caching repeated embeddings or inputs.

> Deploy the whole inference pipeline, not only the weight file.

---
# Week 5 Summary
---

A Transformer system can be understood as:

$$
\boxed{
\text{Raw Data}
\rightarrow
\text{Tokens}
\rightarrow
\text{Embeddings + Position}
\rightarrow
\text{Masked Attention}
\rightarrow
\text{Transformer Blocks}
\rightarrow
\text{Task Head}
\rightarrow
\text{Loss / Metric}
}
$$

Core ideas:

- positional information makes order visible;
- masks define legal information flow;
- attention performs data-dependent token mixing;
- FFNs enrich each token representation;
- model families differ mainly in topology, masks, and objective;
- practical success depends on tokenizer, data, optimization, efficiency, and deployment discipline.

## Week 5 Checklist

You should now be able to answer:

1. What is a token, and how does tokenization differ across modalities?
2. Why does self-attention require positional information?
3. Compare sinusoidal, learned absolute, relative-bias, and RoPE position schemes.
4. Why must masks be applied before softmax?
5. Write the scaled dot-product attention equation.
6. Explain $Q$, $K$, and $V$ using both intuition and tensor shapes.
7. What does multi-head attention add?
8. What are the main components of a Transformer block?
9. Why are residual connections and LayerNorm important?
10. Estimate MHA and FFN parameter counts.
11. Why does attention become expensive for long sequences?
12. Compare BERT, GPT, T5, and ViT by topology, mask, and objective.
13. What does a KV cache do?
14. What is the difference between full fine-tuning, linear probing, and LoRA?
15. Why does dynamic padding improve efficiency?
16. What trade-off does gradient checkpointing make?
17. What is quantization, and what is the QLoRA-style idea?
18. What should be checked when debugging a Transformer training pipeline?